In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path

inputs = Path("/kaggle/input")
print("top-level /kaggle/input entries:", list(inputs.iterdir()))
roots = list((inputs / "nanowm-code").rglob("pyproject.toml"))
if not roots:
    roots = list(inputs.rglob("pyproject.toml"))
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one project root, found {len(roots)}: {roots}")
mounted = roots[0].parent
root = Path("/kaggle/working/project")
root.mkdir(parents=True, exist_ok=True)
shutil.copytree(mounted, root, dirs_exist_ok=True)
for archive in mounted.glob("*.zip"):
    with zipfile.ZipFile(archive) as handle:
        handle.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root))
print("project root:", root)

In [ ]:
subprocess.run([sys.executable, "scripts/preflight_gpu.py"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers"], check=True)
subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q"], check=True)

In [ ]:
# Inference-side: costs Kaggle weekly quota, not the 10-hour training
# budget. 160 trajectories (40 scenes x 4), replacing M1's 8-window set
# that CLAUDE.md's decision log found too undiverse for a meaningful LR
# ranking (see the 2026-08-31 M1 diagnosis entry).
subprocess.run([
    sys.executable, "scripts/preprocess/precompute_dataset.py",
    "--tier", "m3",
    "--out-dir", "/kaggle/working/nanowm_data/m3",
    "--device", "cuda",
], check=True)

In [ ]:
# The checked-in configs/m2_mup_lr_sweep.yaml stays at ticket_hours: null
# (tests/test_m2_lr_sweep.py guards this -- never commit a live ticket
# value for a multi-arm sweep). The real, user-approved value is patched
# in only on this Kaggle copy, never in the local repo.
#
# 0.5 GPU-hours approved 2026-09-04 (CLAUDE.md decision log): ~2.8x the
# 0.181h synthetic-profile estimate (results/m2_profile.json), leaving
# margin for real-dataloader overhead the synthetic profile doesn't
# capture (M1's lesson).
import yaml

cfg_path = Path("configs/m2_mup_lr_sweep.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
cfg["run"]["ticket_hours"] = 0.5
cfg_path.write_text(yaml.safe_dump(cfg))
print(cfg_path.read_text())

In [ ]:
# run_m2_lr_sweep.py enforces its own ticket (scripts.train.check_ticket)
# and logs to the ledger in a finally block regardless of outcome. Exit
# code 2 means the ticket deadline was reached mid-sweep -- the budget
# rule working, not a failure; results up to that point are already
# written incrementally.
result = subprocess.run([
    sys.executable, "scripts/run_m2_lr_sweep.py",
    "--config", str(cfg_path),
])
print(f"sweep exit code: {result.returncode}")
print(Path("budget/ledger.jsonl").read_text())

In [ ]:
import json
print(json.dumps(json.load(open(cfg["output"]["path"])), indent=2))